In [1]:
import os, json
from coadd_functions import coadd_images
from multiprocessing import Pool
from functools import partial
from parallelbar import progress_starmap, progress_map

/data/SOAR/2021A/coadd_functions.py:32: SyntaxWarning: invalid escape sequence '\|'
  for sep in [' & ',' \| ']:
/data/SOAR/2021A/coadd_functions.py:254: SyntaxWarning: invalid escape sequence '\d'
  flat_image = re.findall('master_flat\d\.fits', header['FLATCOR'])[0]


In [2]:
dataset = './2021-10-29/' 
instrument = 'Goodman'
with open(os.path.join(dataset,'imsets.json'),'r') as input_file:
    imagesets=json.load(input_file)
imagesets

{'ASS64_B-Bessel1': {'object': 'ASS64',
  'filter': 'B-Bessel',
  'filter2': 'NO_FILTER',
  'sequence': 6,
  'images': ['./2021-10-29/0275_SO2021B-018_1029.fits',
   './2021-10-29/0276_SO2021B-018_1029.fits'],
  'exptime': [10.0, 10.0],
  'offset': [0.0, 7.16e-05],
  'output': './2021-10-29/ASS64_B-Bessel1.fits'},
 'ASS64_B-Bessel2': {'object': 'ASS64',
  'filter': 'B-Bessel',
  'filter2': 'NO_FILTER',
  'sequence': 6,
  'images': ['./2021-10-29/0285_SO2021B-018_1029.fits',
   './2021-10-29/0286_SO2021B-018_1029.fits',
   './2021-10-29/0287_SO2021B-018_1029.fits'],
  'exptime': [400.0, 400.0, 400.0],
  'offset': [4.11e-05, 4.85e-05, 2.22e-05],
  'output': './2021-10-29/ASS64_B-Bessel2.fits'},
 'ASS64_H-alpha1': {'object': 'ASS64',
  'filter': 'H-alpha',
  'filter2': 'NO_FILTER',
  'sequence': 6,
  'images': ['./2021-10-29/0279_SO2021B-018_1029.fits',
   './2021-10-29/0280_SO2021B-018_1029.fits',
   './2021-10-29/0281_SO2021B-018_1029.fits'],
  'exptime': [120.0, 120.0, 120.0],
  'offse

In [3]:
setimages = [setdict['images'] for _,setdict in imagesets.items()]
outimages = [setdict['output'] for _,setdict in imagesets.items()]
list(zip(setimages,outimages))

[(['./2021-10-29/0275_SO2021B-018_1029.fits',
   './2021-10-29/0276_SO2021B-018_1029.fits'],
  './2021-10-29/ASS64_B-Bessel1.fits'),
 (['./2021-10-29/0285_SO2021B-018_1029.fits',
   './2021-10-29/0286_SO2021B-018_1029.fits',
   './2021-10-29/0287_SO2021B-018_1029.fits'],
  './2021-10-29/ASS64_B-Bessel2.fits'),
 (['./2021-10-29/0279_SO2021B-018_1029.fits',
   './2021-10-29/0280_SO2021B-018_1029.fits',
   './2021-10-29/0281_SO2021B-018_1029.fits'],
  './2021-10-29/ASS64_H-alpha1.fits'),
 (['./2021-10-29/0271_SO2021B-018_1029.fits',
   './2021-10-29/0272_SO2021B-018_1029.fits'],
  './2021-10-29/ASS64_I-Bessel1.fits'),
 (['./2021-10-29/0291_SO2021B-018_1029.fits',
   './2021-10-29/0292_SO2021B-018_1029.fits',
   './2021-10-29/0293_SO2021B-018_1029.fits'],
  './2021-10-29/ASS64_I-Bessel2.fits'),
 (['./2021-10-29/0277_SO2021B-018_1029.fits',
   './2021-10-29/0278_SO2021B-018_1029.fits'],
  './2021-10-29/ASS64_Rc1.fits'),
 (['./2021-10-29/0273_SO2021B-018_1029.fits',
   './2021-10-29/0274_SO2

In [4]:
def coadd_imageset(imageset, multiprocessing=True, **kwargs):

    setimages = [setdict['images'] for _,setdict in imageset.items()]
    outimages = [setdict['output'] for _,setdict in imageset.items()]

    if multiprocessing:
        # result = progress_starmap(partial(coadd_images, **kwargs), list(zip(setimages,outimages)), 
        #                           chunk_size=1, n_cpu=24)
        with Pool() as pool:
            pool.starmap(partial(coadd_images, **kwargs), zip(setimages,outimages),
                         chunksize=1)

    else:
        for imglist,outimage in zip(setimages,outimages):
            print(f"{outimage}:{imglist}")
            coadd_images(imglist, outimage, **kwargs)

In [5]:
coadd_imageset(imagesets, multiprocessing=True, instrument='Goodman')